# Setup — MECTESIS en Vertex AI Workbench

Ejecutar cada celda en orden. Solo la primera vez, en una instancia nueva.

**Repositorio:** https://github.com/DavidGuzzi/MECTESIS  
**Tiempo estimado:** ~5-8 min total

## 1 — Clonar el repositorio

Si ya estás dentro del repo clonado (ej. abriste este notebook desde JupyterLab de la instancia), saltear esta celda.

In [ ]:
import os, subprocess, sys
from pathlib import Path

# Detectar si ya estamos dentro del repo
in_repo = Path("mectesis").is_dir() and Path("pyproject.toml").exists()

if in_repo:
    print(f"Ya dentro del repo: {Path.cwd()}")
else:
    home = Path.home()
    repo_dir = home / "MECTESIS"
    if not repo_dir.exists():
        subprocess.run(["git", "clone", "https://github.com/DavidGuzzi/MECTESIS.git", str(repo_dir)], check=True)
    else:
        subprocess.run(["git", "-C", str(repo_dir), "pull"], check=True)
    os.chdir(repo_dir)
    print(f"Directorio de trabajo: {Path.cwd()}")

print(f"Python: {sys.version}")

## 2 — Instalar dependencias

Instala `mectesis` en modo editable junto con todas sus dependencias.

In [ ]:
import subprocess, sys

# pip install -e . instala mectesis + dependencias declaradas en pyproject.toml
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", ".", "--quiet"],
    capture_output=True, text=True
)
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])
else:
    print("Instalación completada")
    if result.stdout.strip():
        print(result.stdout[-500:])

## 3 — Verificar imports

In [ ]:
from mectesis.dgp import ARpDGP, MAqDGP, ARMApqDGP, RandomWalk
from mectesis.models.arima import ARIMAModel
from mectesis.simulation import MonteCarloEngine
from mectesis.metrics import BiasVarianceMSE
print("mectesis importado correctamente")

## 4 — Cargar Chronos-2

La primera vez descarga ~1.5 GB desde Hugging Face. Las siguientes usa caché.

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

from mectesis.models import ChronosModel
chronos = ChronosModel(device=device)
print("Chronos-2 listo")

## 5 — Crear directorio de resultados

In [ ]:
from pathlib import Path
results_dir = Path("results/univariate_v3")
results_dir.mkdir(parents=True, exist_ok=True)
print(f"Directorio listo: {results_dir.resolve()}")

## 6 — Smoke test (opcional)

Verifica el pipeline completo con R=5, T=50, H=6.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
from mectesis.dgp import ARpDGP
from mectesis.models.arima import ARIMAModel
from mectesis.simulation import MonteCarloEngine

dgp = ARpDGP(phis=[0.5], sigma=1.0, seed=42)
models = [ARIMAModel((1, 0, 0))]
engine = MonteCarloEngine(dgp, models, seed=42)
res = engine.run_monte_carlo(n_sim=5, T=50, horizon=6, dgp_params={}, verbose=True)
print("RMSE h=1:", round(float(res["ARIMA(1, 0, 0)"]["rmse"].iloc[0]), 4))
print("Smoke test OK")

---
## Listo para ejecutar el notebook principal

Abrir y ejecutar:

```
notebooks/experimentos_univariados_v3_cloud.ipynb
```

**Kernel → Restart Kernel and Run All Cells**  

Los resultados se guardan en `results/univariate_v3/` (CSV por experimento + archivo `.log`).  
Podés cerrar el navegador — el kernel sigue corriendo en el servidor.